In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("pandas", pd.__version__)
print("numpy", np.__version__)


pandas 3.0.6
numpy 2.5.4


In [3]:
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
BASE = "https://raw.githubusercontent.com/Tennismylife/TML-Database/master/{}.csv"

for year in range(2000, 2027):
    path = RAW / f"tml_{year}.csv"
    if not path.exists():
        pd.read_csv(BASE.format(year)).to_csv(path, index=False)
        print("downloaded", year)
    else:
        print("cached    ", year)

downloaded 2000
downloaded 2001
downloaded 2002
downloaded 2003
downloaded 2004
downloaded 2005
downloaded 2006
downloaded 2007
downloaded 2008
downloaded 2009
downloaded 2010
downloaded 2011
downloaded 2012
downloaded 2013
downloaded 2014
downloaded 2015
downloaded 2016
downloaded 2017
downloaded 2018
downloaded 2019
downloaded 2020
downloaded 2021
downloaded 2022
downloaded 2023
downloaded 2024
downloaded 2025
downloaded 2026


In [4]:
df = pd.concat(
    [pd.read_csv(RAW / f"tml_{y}.csv") for y in range(2000, 2027)],
    ignore_index=True,
)
df["date"] = pd.to_datetime(df["tourney_date"], format="%Y%m%d")

print("shape:", df.shape)
print("последняя дата:", df["date"].max())
print(df.groupby(df["date"].dt.year).size().tail(4))
print(df[df["tourney_name"] == "US Open"].groupby(df["date"].dt.year).size().tail(4))

shape: (78091, 51)
последняя дата: 2026-01-17 00:00:00
date
2023    2995
2024    3159
2025    2861
2026     137
dtype: int64
date
2022    127
2023    127
2024    127
2025    127
dtype: int64


In [5]:
k = pd.read_csv(RAW / "atp_tennis.csv")
k["Date"] = pd.to_datetime(k["Date"])

print("shape:", k.shape)
print("колонки:", k.columns.tolist())
print("последняя дата:", k["Date"].max())
print(k[k["Tournament"].str.contains("US Open", na=False)].groupby(k["Date"].dt.year).size().tail(4))

shape: (68872, 17)
колонки: ['Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Player_1', 'Player_2', 'Winner', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2', 'Score']
последняя дата: 2026-10-06 00:00:00
Date
2023    120
2024    119
2025    122
2026    125
dtype: int64


In [6]:
print(k.head(3).T)
print()
print("доля Winner == Player_1:", round((k["Winner"] == k["Player_1"]).mean(), 3))
print()
print("пропуски:")
print(k.isna().sum()[lambda s: s > 0])
print()
print("Odd <= 1:", ((k["Odd_1"] <= 1) | (k["Odd_2"] <= 1)).sum())
print()
us26 = k[k["Tournament"].str.contains("US Open", na=False) & (k["Date"].dt.year == 2026)]
print(us26["Round"].value_counts())
print("даты US Open 2026:", us26["Date"].min(), "-", us26["Date"].max())

                                             0  \
Tournament  Australian Hardcourt Championships   
Date                       2000-01-03 00:00:00   
Series                           International   
Court                                  Outdoor   
Surface                                   Hard   
Round                                1st Round   
Best of                                      3   
Player_1                            Arthurs W.   
Player_2                          Gambill J.M.   
Winner                            Gambill J.M.   
Rank_1                                     105   
Rank_2                                      58   
Pts_1                                       -1   
Pts_2                                       -1   
Odd_1                                     -1.0   
Odd_2                                     -1.0   
Score                              6-3 6-7 4-6   

                                             1  \
Tournament  Australian Hardcourt Championships   

In [7]:
k = k.copy()

# -1 -> NaN
for c in ["Rank_1", "Rank_2", "Pts_1", "Pts_2"]:
    k.loc[k[c] < 0, c] = np.nan
for c in ["Odd_1", "Odd_2"]:
    k.loc[k[c] <= 1, c] = np.nan

# доля пропусков по годам
miss = k[["Rank_1", "Pts_1", "Odd_1"]].isna().groupby(k["Date"].dt.year).mean().round(2)
print(miss)

# Date - это дата матча или дата начала турнира?
us25 = k[k["Tournament"].str.contains("US Open", na=False) & (k["Date"].dt.year == 2025)]
print()
print(us25.groupby("Round")["Date"].agg(["min", "max"]))

      Rank_1  Pts_1  Odd_1
Date                      
2000     0.0   1.00   1.00
2001     0.0   1.00   0.15
2002     0.0   1.00   0.06
2003     0.0   1.00   0.02
2004     0.0   1.00   0.02
2005     0.0   0.58   0.01
2006     0.0   0.00   0.02
2007     0.0   0.00   0.02
2008     0.0   0.00   0.01
2009     0.0   0.00   0.01
2010     0.0   0.00   0.00
2011     0.0   0.00   0.00
2012     0.0   0.00   0.00
2013     0.0   0.00   0.00
2014     0.0   0.00   0.00
2015     0.0   0.00   0.00
2016     0.0   0.00   0.00
2017     0.0   0.00   0.00
2018     0.0   0.00   0.00
2019     0.0   0.00   0.00
2020     0.0   0.00   0.00
2021     0.0   0.00   0.00
2022     0.0   0.00   0.00
2023     0.0   0.00   0.00
2024     0.0   0.00   0.00
2025     0.0   0.00   0.00
2026     0.0   0.00   0.00

                     min        max
Round                              
1st Round     2025-08-24 2025-08-27
2nd Round     2025-08-27 2025-08-29
3rd Round     2025-08-29 2025-08-31
4th Round     2025-08-31 2025-09-02


In [8]:
%%writefile ../src/loader.py
import numpy as np
import pandas as pd

ROUND_ORDER = {
    "1st Round": 1, "2nd Round": 2, "3rd Round": 3, "4th Round": 4,
    "Quarterfinals": 5, "Semifinals": 6, "The Final": 7,
}

RENAME = {
    "Tournament": "tourney", "Date": "date", "Series": "series",
    "Court": "court", "Surface": "surface", "Round": "round",
    "Best of": "best_of", "Player_1": "p1", "Player_2": "p2",
    "Winner": "winner", "Rank_1": "rank1", "Rank_2": "rank2",
    "Pts_1": "pts1", "Pts_2": "pts2", "Odd_1": "odd1", "Odd_2": "odd2",
    "Score": "score",
}


def load_matches(path):
    """Читает Kaggle ATP, чистит и сортирует по времени."""
    m = pd.read_csv(path, parse_dates=["Date"]).rename(columns=RENAME)

    # пропуски записаны как -1 (очки/ранг) и -1.0 (коэффициенты)
    for c in ["rank1", "rank2", "pts1", "pts2"]:
        m.loc[m[c] < 0, c] = np.nan
    for c in ["odd1", "odd2"]:
        m.loc[m[c] <= 1, c] = np.nan

    # победитель должен быть одним из двух игроков
    ok = (m["winner"] == m["p1"]) | (m["winner"] == m["p2"])
    if not ok.all():
        raise ValueError(f"winner не совпадает с игроками в {(~ok).sum()} строках")

    m["y"] = (m["winner"] == m["p1"]).astype(int)   # 1, если выиграл Player_1
    m["round_order"] = m["round"].map(ROUND_ORDER).fillna(0).astype(int)

    # стабильная сортировка: дата, затем раунд
    m = m.sort_values(["date", "round_order"], kind="stable").reset_index(drop=True)
    return m

Writing ../src/loader.py


In [9]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")
from loader import load_matches

m = load_matches(RAW / "atp_tennis.csv")

print("shape:", m.shape)
print()
print(m["round"].value_counts())
print()
print("дубликаты:", m.duplicated(["date", "tourney", "p1", "p2"]).sum())
print("доля y=1:", round(m["y"].mean(), 3))
print()
print(m[["date", "tourney", "round", "p1", "p2", "y", "odd1", "odd2"]].tail(5))

shape: (68872, 19)

round
1st Round        31513
2nd Round        19234
Quarterfinals     6504
3rd Round         4882
Semifinals        3323
The Final         1688
4th Round         1302
Round Robin        426
Name: count, dtype: int64

дубликаты: 2
доля y=1: 0.5

            date                          tourney          round          p1  \
68867 2026-10-04  Japan Open Tennis Championships  Quarterfinals  Lehecka J.   
68868 2026-10-04  Japan Open Tennis Championships  Quarterfinals    Munar J.   
68869 2026-10-05  Japan Open Tennis Championships     Semifinals  Lehecka J.   
68870 2026-10-05  Japan Open Tennis Championships     Semifinals    Munar J.   
68871 2026-10-06  Japan Open Tennis Championships      The Final  Lehecka J.   

                p2  y  odd1  odd2  
68867   Vallejo D.  1  1.33  3.40  
68868   Jacquet K.  1  1.67  2.20  
68869  Vacherot V.  1  1.50  2.63  
68870   Alcaraz C.  0  7.00  1.10  
68871   Alcaraz C.  0  3.75  1.29  
